In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DecimalType,
    TimestampType
)

kafka_order_schema = StructType([
    StructField("rating_key", StringType(), True),
    StructField("restaurant_key", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("driver_key", StringType(), True),
    StructField("order_date", TimestampType(), True),
    StructField("user_key", StringType(), True),
    StructField("total_amount", DecimalType(12,2), True),
    StructField("payment_key", StringType(), True),
    StructField("dt_current_timestamp", TimestampType(), True),
])

# schemaEvolutionMode
- addNewColumns (Padrão): Stream fails. New columns are added to the schema. Existing columns do not evolve data types.
- rescue Schema is never evolved and stream does not fail due to schema changes. All new columns are recorded in the rescued data column. 
- failOnNewColumns Stream fails. Stream does not restart unless the provided schema is updated, or the offending data file is removed.
- none Does not evolve the schema, new columns are ignored, and data is not rescued unless the rescuedDataColumn option is set. Stream does not fail due to schema changes.

# trigger
- availableNow : Processes all the available data as mutiple micro-batch and stops 
- processingTime : Micro batchs starts at the user specified interval

In [0]:
from pyspark.sql.functions import current_timestamp

SOURCE_DIR = "/Volumes/uber/ingestion/kafka/orders"
CHECKPOINT_DIR = "/Volumes/uber/ingestion/kafka/checkpoint/"
SCHEMA_DIR = "/Volumes/uber/ingestion/kafka/schema/"
BADREC_DIR = "/Volumes/uber/ingestion/kafka/badrecords/"

(
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_DIR)
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss'Z'")
    .option("cloudFiles.maxBytesPerTrigger", "5g")
    .option("badRecordsPath", BADREC_DIR)
    .load(SOURCE_DIR)
    .withColumn("ingest_time",current_timestamp())
    .writeStream
    .option("checkpointLocation", CHECKPOINT_DIR)
    .trigger(availableNow=True)
    .toTable("uber.ingestion.kafka_orders_loader")
)